# Modelos tradicionais

Treina o pipeline clássico do Fake.br-Corpus e persiste os artefatos no repositório, em `results/modelos/`. Este notebook não usa GPU: TF-IDF, Linear SVC e regressão logística rodam em CPU.

Os hiperparâmetros e o split são os mesmos de `main.ipynb` (`test_size=0.2`, `random_state=42`). O texto passa por `text_cleaning` antes da vetorização.

Arquivos gravados em `results/modelos/`:

- `vectorizer_tfidf.joblib`
- `modelo_svc.joblib`
- `modelo_lr.joblib`

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'requirements.txt').exists():
    REPO_ROOT = Path('/content/classificador-fake-br')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'pt_core_news_sm'], check=True)

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Dependências prontas para o pipeline clássico.')

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path('/content/classificador-fake-br')
if not (REPO_ROOT / 'src').exists():
    raise FileNotFoundError(f'Repositório não encontrado a partir de {Path.cwd()}')
sys.path.insert(0, str(REPO_ROOT))

from src.pipeline import conecta_drive
from src.preprocessing import text_cleaning
from src.evaluation import (
    MODELOS_CLASSICOS_DIR,
    persistir_modelos_classicos,
    relatorio_completo,
    tabela_resultados,
    treinar_lr,
    treinar_svc,
)
from src.visualization import plot_confusion_matrix

print(f'Repositório: {REPO_ROOT}')

## Drive e corpus

No Colab, a célula monta o Google Drive. No computador, ela usa a pasta sincronizada pelo Google Drive para desktop. Se essa pasta tiver outro caminho, defina `FAKE_BR_DRIVE_ROOT` com o diretório `My Drive`.

In [ ]:
DRIVE_ROOT = conecta_drive()
DATASET_DIR = DRIVE_ROOT / 'Fake.br-Corpus'
CSV_PATH = DATASET_DIR / 'fake.br-full_texts.csv'
MODELOS_DIR = MODELOS_CLASSICOS_DIR

if not CSV_PATH.is_file():
    raise FileNotFoundError(f'CSV do corpus não encontrado: {CSV_PATH}')

df = pd.read_csv(CSV_PATH)
if 'texto_completo' not in df.columns or 'label' not in df.columns:
    raise ValueError(f'Colunas esperadas ausentes. Encontradas: {list(df.columns)}')

print(f'Drive: {DRIVE_ROOT}')
print(f'Corpus: {CSV_PATH} ({len(df):,} notícias)')
print(f'Modelos serão salvos no repositório: {MODELOS_DIR}')

## Pré-processamento, treino e persistência

A limpeza reutiliza `text_cleaning`. O vetorizador é ajustado só no treino. `treinar_svc` e `treinar_lr` ajustam os classificadores, e `persistir_modelos_classicos` grava os três artefatos em `results/modelos/`, dentro do repositório.

In [ ]:
df['clean_text'] = df['texto_completo'].apply(text_cleaning)

X = df['clean_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

modelo_svc, pred_svc = treinar_svc(X_train_tfidf, y_train, X_test_tfidf)
modelo_lr, pred_lr = treinar_lr(X_train_tfidf, y_train, X_test_tfidf)

relatorio_completo(y_test, pred_svc, 'Linear SVC')
relatorio_completo(y_test, pred_lr, 'Logistic Regression')
df_resultados = tabela_resultados(
    y_test,
    [(pred_svc, 'Linear SVC'), (pred_lr, 'Logistic Regression')],
)
print(df_resultados.to_string(index=False))

caminhos = persistir_modelos_classicos(MODELOS_DIR, vectorizer, modelo_svc, modelo_lr)
print('Artefatos gravados:')
for caminho in caminhos.values():
    print(f'- {caminho}')

In [ ]:
plot_confusion_matrix(y_test, pred_svc)
plot_confusion_matrix(y_test, pred_lr)